# 5. Events, logs, and checkpoints

Events are append-only observations; checkpoints are mutable continuation state. Events support audit and evaluation. A checkpoint lets approval resume the exact pending action without asking the model to recreate it.

## Before you begin

**Required — all students:** run mock mode first. **Choose one:** repeat provider lessons with OpenRouter when configured. The real MCP stdio cell requires the pinned Day 5 SDK; fake MCP is the fallback.

### Learning outcomes

Distinguish events from checkpoints and prove a durable checkpoint can resume after runtime reconstruction.

Architecture reference: [Day 5 diagrams D16](../../diagrams/source/day_05.md).

### Expected observation

Approval state survives a new JSON checkpoint-store instance and clears after resolution. Exact IDs, timing, and live wording will vary.

## Concept briefing

## Events and checkpoints

Events are append-only observations such as run started, model completed, policy decided
and tool completed. A trace groups events belonging to one run. A checkpoint stores
continuation state so a paused run can resume.

A checkpoint is not an audit log, and an event log is not enough to resume execution.
Durable approval needs the exact pending action and a stable run identifier. Sensitive
arguments should be redacted or omitted from telemetry where possible.

## Retries, timeouts and retry budgets

Network calls fail. A runtime should apply a timeout and may retry transient failures such
as temporary rate limits. Retries must be bounded and recorded. Exponential backoff with
jitter helps avoid many clients retrying simultaneously.

Do not retry every failure. Invalid arguments, policy denial and most authentication
errors will not improve on repetition. Consequential tools require an idempotency strategy
before automatic retry. The runtime should have both a step budget and a retry budget so
one failing provider does not consume unlimited time or credit.

## Cost attribution

Record model, prompt/configuration version, input tokens, output tokens, reasoning tokens,
estimated cost and run ID. This makes it possible to compare agents and enforce classroom
budgets. Cost belongs to the complete run, including retries and specialist calls, not
only the final response.

The included API credit is a controlled learning resource. Mock mode should be used while
debugging application logic; live calls should be used when model behavior is the subject
of the exercise.


In [ ]:
from pathlib import Path
import sys,json
DAY=Path.cwd()
if (DAY/"day_05_ai_harness").exists(): DAY=DAY/"day_05_ai_harness"
elif DAY.name=="notebooks": DAY=DAY.parent
if not (DAY/"src"/"mini_harness").exists(): raise RuntimeError("Launch Jupyter from the repository, day folder, or notebooks folder.")
sys.path.insert(0,str(DAY/"src"))
from mini_harness import *
def load_config(name):
    raw=json.loads((DAY/"configs"/f"{name}.json").read_text(encoding="utf-8"))
    raw["model"]=ModelConfig(**raw["model"])
    return AgentConfig(**raw)
print("Day folder:",DAY)

In [ ]:
events=EventStore(); checkpoint_dir=DAY/"data"/"demo_checkpoints"
checkpoints=JSONCheckpointStore(checkpoint_dir)
runtime=HarnessRuntime(build_demo_registry(),MockModel(),events,checkpoints)
cfg=load_config("task_agent"); paused=runtime.run(cfg,"Send the synthetic update")
for event in events.get(paused.run_id): print(event)
print("Saved state:",checkpoints.load(paused.run_id))
# Simulate a restart by constructing a new runtime and checkpoint-store object.
resumed_runtime=HarnessRuntime(build_demo_registry(),MockModel(),events,JSONCheckpointStore(checkpoint_dir))
done=resumed_runtime.resume(paused.run_id,cfg,approved=True)
print("Final:",done.status,done.output)
print("Checkpoint cleared:",checkpoints.load(paused.run_id))

## Persisting locally

Pass a JSONL path to `EventStore` for durable logs. The teaching checkpoint store is in-memory and transparent; replacing it with SQLite is a useful extension. Do not log secrets, private prompts, or hidden reasoning. LangSmith export remains optional and synthetic-only.

## Your turn

Inspect the checkpoint file, reconstruct the runtime, resume, and verify the event order.

## Recap

Events explain history; checkpoints preserve continuation state. Name one responsibility that deliberately remains application-specific.